# Review and provenance

Placeholder notebook (to be extended): where each value came from, which cells the pipeline flagged, and what the owner's review changed. Reads only the dataset and `config/review/volume1_verdicts.csv`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))  # notebooks/ (for pgdata)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import pgdata
arcs, towers = pgdata.arcs(), pgdata.towers()
len(arcs), len(towers)

In [ ]:
both = pd.concat([arcs.assign(table="5.2 arcs"), towers.assign(table="5.3 towers")], ignore_index=True)
both["source"] = np.where(both["value_source"].str.startswith("owner review"), "owner review", both["value_source"])
pd.crosstab(both["source"], both["status"])

## Flags by arc

The pipeline's doubt flags are kept in the data; review covered only cells that disagreed with the reference.

In [ ]:
both.assign(group=both["arc_m"].astype("Int64").astype(str).replace("<NA>", "towers")) \
    .groupby("group").agg(cells=("cell_id", "size"), flagged=("flagged", "sum"), reviewed=("reviewed", "sum"))

## The owner's verdicts

In [ ]:
import pathlib
v = pd.read_csv(pathlib.Path(pgdata.REPO, "config/review/volume1_verdicts.csv"), dtype=str, keep_default_na=False)
v.groupby("group").agg(cells=("cell_id", "size"),
                       mistral_right=("mistral_right", lambda s: (s == "True").sum()),
                       sonnet_right=("sonnet_right", lambda s: (s == "True").sum()))

In [ ]:
# cells where the printed value differs from Mistral's reading
v[v.mistral_right == "False"][["cell_id", "mistral_token", "sonnet_token", "token_as_printed", "basis"]]